In [1]:
import pandas as pd
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

# Load dataset
df = pd.read_csv("customer_data.csv")

# Features
X = df[['Age','AnnualIncome','SpendingScore']]

# Target
y = (df["YearsCustomer"] > 7).astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators",50,200)

    max_depth = trial.suggest_int("max_depth",2,10)

    min_samples_split = trial.suggest_int("min_samples_split",2,10)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    ).mean()

    return score

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)

print("Best Accuracy:", study.best_value)
print("Best Parameters:", study.best_params)

results = study.trials_dataframe()

results.to_csv("tuning_results.csv", index=False)

with open("best_parameters.txt","w") as f:
    f.write(str(study.best_params))

/Users/sreedev/Desktop/meetmux/Task17_Hyperparameter_Tuning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-07 11:49:40,166] A new study created in memory with name: no-name-e0788e69-4c9b-4408-b95a-ff30d49ccaa6
[I 2026-07-07 11:49:40,385] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 116, 'max_depth': 3, 'min_samples_split': 5}. Best is trial 0 with value: 1.0.
[I 2026-07-07 11:49:40,644] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 156, 'max_depth': 2, 'min_samples_split': 9}. Best is trial 0 with value: 1.0.
[I 2026-07-07 11:49:40,868] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 134, 'max_depth': 3, 'min_samples_split': 8}. Best is trial 0 with value: 1.0.
[I 2026-07-07 11:49:41,023] Trial 3 finished with value: 1.0 and pa

Best Accuracy: 1.0
Best Parameters: {'n_estimators': 116, 'max_depth': 3, 'min_samples_split': 5}


In [5]:
import os
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

os.makedirs("images", exist_ok=True)

# Optimization History
plot_optimization_history(study)
plt.tight_layout()
plt.savefig("images/optimization_history.png")
plt.close()

# Parameter Importance
try:
    plot_param_importances(study)
    plt.tight_layout()
    plt.savefig("images/parameter_importance.png")
    plt.close()
except RuntimeError:
    print("Parameter importance could not be generated because all trials achieved the same score.")

# Model Performance
plt.figure(figsize=(6,4))
plt.bar(["Best Accuracy"], [study.best_value])
plt.title("Best Model Performance")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig("images/model_performance.png")
plt.close()

print("Images saved successfully!")

Parameter importance could not be generated because all trials achieved the same score.
Images saved successfully!


/var/folders/4c/kd1ssmkd0cx_2fg5kkp5vkb40000gn/T/ipykernel_70840/2857432923.py:11: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  plot_optimization_history(study)
/var/folders/4c/kd1ssmkd0cx_2fg5kkp5vkb40000gn/T/ipykernel_70840/2857432923.py:18: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  plot_param_importances(study)


In [ ]:
import numpy as np

np.random.seed(42)

# Add small random noise to features
X = X + np.random.normal(0, 0.2, X.shape)